# The HALD Knowledge Graph

This section has been inspired by the work of Robert Haas on Biomedical Knowledge Graphs which can be found at 
https://github.com/robert-haas/awesome-biomedical-knowledge-graphs/tree/main
The source of the data is the webpage on [Figshare](https://figshare.com/articles/dataset/HALD_a_human_aging_and_longevity_knowledge_graph_for_precision_gerontology_and_geroscience_analyses/22828196). This include versions in JSON and CSV. The CSV versions are structured for a particular package which we will not be using, so we will use the json packages which are more general.

First we'll create a data directory.

In [14]:
import os
Download = False
datadir = "HALD_Dataset"
if not os.path.exists(datadir):
    os.mkdir(datadir)

Now we define the list of the files that we want to download. We'll define a *list* of *tuples*, with each tuple representing one of the files that we want to fetch, specifying three things:

* The name that we want the file to be called.
* The URL from where it will be downloaded.
* The MD5 checksum which will allow us to verify the downloaded file's integrity.

In [15]:
# List of files to download
filelist = [
    ("Entity_info.json", "https://figshare.com/ndownloader/files/43612509", '1746cde24a1bac0460f1ccf646608cc9'),
    ("Literature_Info.json", "https://figshare.com/ndownloader/files/43612512", "10b78e8ec30f5b85f2a58d8fe24f056b"),
    ("Longevity_Biomarkers.json", "https://figshare.com/ndownloader/files/43612497", "0dbd9c3f8474dc3cd744ed38af460d75"),
    ("Relation_Info.json", "https://figshare.com/ndownloader/files/43612506", "0c1fa199269adc58f64ad4d5b9fd87b9"),
    ("Aging_Biomarkers.json", "https://figshare.com/ndownloader/files/43612503", "abd0eb6cb7295ae500c5d676b7797324")
]

Now we can download the files. For each file in `filelist` we will:

* Download the file from the URL.
* If the download request indicates that the download is unsuccessful, print an error.
* If the download is successfull, verify the checksum and if that is correct, write the file to disk in `datadir`

In [16]:
import requests
import hashlib

if Download:
    for f in filelist:
        response = requests.get(f[1])
        file_Path = datadir + "/" + f[0]
        if response.status_code != 200:
            print('Failed to download file {f[0]} from {f[1]}')
        else:
            m = hashlib.md5()
            m.update(response.content)
            if m.hexdigest() == f[2]:
                print(f"SUCCESS: File {f[0]} downloaded from {f[1]} with correct checksum {f[2]}")
                with open(file_Path, 'wb') as file:
                    file.write(response.content)
            else:
                print(f"ERROR: File {f[0]} downloaded from {f[1]} with incorrect checksum {m.hexdigest()} (should be {f[2]})")            


## What does the data look like?

Let us inspect these files. The two key files here are those containing the *entities* (nodes) and the *edges* (relations).

In [17]:
import json

def load_json(fname):
    with open(fname, 'rb') as file:
        return json.load(file)

Entity_info = load_json(f"{datadir}/{filelist[0][0]}")
Literature_info = load_json(f"{datadir}/{filelist[1][0]}")
Longevity_Biomarkers = load_json(f"{datadir}/{filelist[2][0]}")
Relation_info = load_json(f"{datadir}/{filelist[3][0]}")
Aging_Biomarkers = load_json(f"{datadir}/{filelist[4][0]}")

What are these new data structures?

In [18]:
print(type(Entity_info))
print(type(Relation_info))

<class 'dict'>
<class 'dict'>


How big are they?

In [19]:
print(f"There are {len(Entity_info)} Entities and {len(Relation_info)} Relations")


There are 12257 Entities and 116495 Relations


Take a better look at the dataset: what's in it?

In [20]:
# Get the entity keys
first_five_keys = list(Entity_info.keys())[0:5]
print(first_five_keys)
for k in first_five_keys:
    print(f"{k}: {Entity_info[k]}")
    if len(Entity_info[k]) != 1:
        print(f"Entity {k} is a multilist")

['MLH1', 'CD4', 'INS', 'MAPT', 'MYC']
MLH1: [{'entity': 'MLH1', 'type': 'Gene', 'PMID': ['12612901', '30275527', '25311944', '22936446', '19949675', '22406557', '23240038', '11325821', '21042749', '25556597', '17556535', '29425284', '22740444', '10954253', '37380216'], 'official full name': 'mutL homolog 1', 'sentence': [['Most such cancers have the CpG island methylator phenotype (CIMP+) with methylation and transcriptional silencing of the mismatch repair gene MLH1.'], ['Our group recently demonstrated that aging human HSCs accumulate microsatellite instability coincident with loss of MLH1, a DNA Mismatch Repair (MMR) protein, which could reasonably predispose to radiation-induced HSC malignancies.', 'In addition, whole-exome sequencing analysis revealed high SNVs and INDELs in lymphomas being driven by loss of Mlh1 and frequently mutated genes had a strong correlation with human leukemias.'], ['ARID1A loss was observed in 9% (22/257) of the cohort: 24% of MMR-deficient tumors (14/59

So each entry is a list of length 1, and that entry contains a dictionary of the node's attributes. Those attributes are sometimes lists.

Let's get a list of the types of the entities create them

In [21]:
EntityTypes = set()
for k in Entity_info.keys():
    EntityTypes.add(Entity_info[k][0]['type'])

from owlready2 import *
onto = get_ontology("http://www.dummy.info/new.owl")

with onto:
    EntityClasses = dict()
    for entity in EntityTypes:
        EntityClasses[entity] = type(entity, (Thing,), dict())
        print(f'Created entity class {EntityClasses[entity]}')


Created entity class new.Lipid
Created entity class new.RNA
Created entity class new.Mutation
Created entity class new.Carbohydrate
Created entity class new.Protein
Created entity class new.Disease
Created entity class new.Toxin
Created entity class new.Gene
Created entity class new.Peptide
Created entity class new.Pharmaceutical Preparations


Now do the same with the attributes. We loop over the entities identifying new attributes and their domain and range. Here we need to be mindful of a limitation: Owlready2 does not support all datatyoe as ranges for the DataProperty class. We will ignore those attributes. The list of valid types for the range can be found at https://owlready2.readthedocs.io/en/latest/properties.html

In [22]:
AttributeType = dict()

for entity in Entity_info.keys():
    for attribute in Entity_info[entity][0].keys():
        ClassOfType = EntityClasses[Entity_info[entity][0]['type']]
        TypeOfAttribute = type(Entity_info[entity][0][attribute])
        # Can't have list as a type in owl so need to get the type of the list elements instead
        if attribute not in AttributeType.keys():
            AttributeType[attribute] = {'domain':  {ClassOfType}, 'range': {TypeOfAttribute}}
        else:
            AttributeType[attribute]['domain'].add(ClassOfType)
            AttributeType[attribute]['range'].add(TypeOfAttribute)

print(EntityTypes)
AttributesToBeRemoved = list()
for attribute in AttributeType.keys():
    AttributeType[attribute]['domain'] = list(AttributeType[attribute]['domain'])
    AttributeType[attribute]['range'] = list(AttributeType[attribute]['range'])
    if any(a not in [str,int,float,bool] for a in AttributeType[attribute]['range']):
           AttributesToBeRemoved.append(attribute)
    print(f"{attribute}: {AttributeType[attribute]}")

for attribute in AttributesToBeRemoved:
    print(f"Removing attribute {attribute} because of incompatible type")
    AttributeType.pop(attribute)

print("Remaining attributes are:")
for attribute in AttributeType.keys():
    print(f"{attribute}: {AttributeType[attribute]}")



{'Lipid', 'RNA', 'Mutation', 'Carbohydrate', 'Protein', 'Disease', 'Toxin', 'Gene', 'Peptide', 'Pharmaceutical Preparations'}
entity: {'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new.Mutation, new.Disease, new.Lipid, new.Protein, new.Carbohydrate, new.Peptide, new.Toxin], 'range': [<class 'str'>]}
type: {'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new.Mutation, new.Disease, new.Lipid, new.Protein, new.Carbohydrate, new.Peptide, new.Toxin], 'range': [<class 'str'>]}
PMID: {'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new.Mutation, new.Disease, new.Lipid, new.Protein, new.Carbohydrate, new.Peptide, new.Toxin], 'range': [<class 'list'>]}
official full name: {'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new.Mutation, new.Disease, new.Lipid, new.Protein, new.Carbohydrate, new.Peptide, new.Toxin], 'range': [<class 'NoneType'>, <class 'str'>]}
sentence: {'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new

Now create the attribute classes

In [23]:
with onto:
    AttributeClasses = dict()
    print(AttributeType.keys())
    for attribute in AttributeType.keys():
        print(attribute)
        print(AttributeType[attribute])
        AttributeClasses[attribute] = type(attribute, (DataProperty,), AttributeType[attribute])
        print(f'Created attribute class {AttributeClasses[attribute]}')

dict_keys(['entity', 'type', 'numbers of articles', 'alias names', 'description', 'mutation position', 'mutation alleles', 'MeSH ID', 'relation', 'aging biomarker', 'longevity biomarker'])
entity
{'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new.Mutation, new.Disease, new.Lipid, new.Protein, new.Carbohydrate, new.Peptide, new.Toxin], 'range': [<class 'str'>]}
Created attribute class new.entity
type
{'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new.Mutation, new.Disease, new.Lipid, new.Protein, new.Carbohydrate, new.Peptide, new.Toxin], 'range': [<class 'str'>]}
Created attribute class new.type
numbers of articles
{'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new.Mutation, new.Disease, new.Lipid, new.Protein, new.Carbohydrate, new.Peptide, new.Toxin], 'range': [<class 'int'>]}
Created attribute class new.numbers of articles
alias names
{'domain': [new.RNA, new.Pharmaceutical Preparations, new.Gene, new.Mutation, new.Disease, new.Lipi

Now to populate the entities. There is one small issue here: there is an entry in the data "entity": "Disease". This conflicts the name of the one of the types, and hence of one of the EntityClasses. We have to trap for it and rename it.

In [ ]:
Nodes = dict()
with onto:
    for entity in Entity_info.keys():
        # Remove invalid attributes
        NodeAttributes = Entity_info[entity][0]
        for attribute in AttributesToBeRemoved:
            NodeAttributes.pop(attribute)
        NodeName = NodeAttributes.pop('entity')
        NodeKey = entity
        if entity == "Disease":
            print(AttributeType.keys())
            print(NodeAttributes)
            NodeName = 'DiseaseNode'
            NodeKey = NodeName
        Nodes[NodeKey] = EntityClasses[NodeAttributes['type']](name=NodeName)
        for k in NodeAttributes.keys():
            getattr(Nodes[NodeKey],k).append(NodeAttributes[k])

dict_keys(['entity', 'type', 'numbers of articles', 'alias names', 'description', 'mutation position', 'mutation alleles', 'MeSH ID', 'relation', 'aging biomarker', 'longevity biomarker'])
{'type': 'Disease', 'numbers of articles': 275, 'alias names': '', 'description': 'A definite pathologic process with a characteristic set of signs and symptoms.', 'mutation position': '', 'mutation alleles': '', 'MeSH ID': 'D004194', 'relation': True, 'aging biomarker': False, 'longevity biomarker': False}


Let us now create the edges. First check the structure of the data

In [30]:
for k in Relation_info.keys():
    print(Relation_info[k])

{'source entity': 'Pulmonary Disease, Chronic Obstructive', 'relationship': 'defined', 'target entity': 'Inflammation', 'sentence': ['(1) Background: Chronic obstructive pulmonary disease (COPD) is defined as an inflammatory disorder that presents an increasingly prevalent health problem.'], 'source': ['COPD'], 'target': ['inflammatory disorder'], 'source type': ['Disease'], 'target type': ['Disease'], 'PMID': ['30781849'], 'DP': ['2019 Feb 13'], 'date': [20190213], 'TI': ['Chronic Obstructive Pulmonary Disease as a Main Factor of Premature Aging.'], 'TA': ['Int J Environ Res Public Health'], 'IF': [0.0], 'IF5': [0.0], 'method': ['deep learning', 'shortest path']}
{'source entity': 'Anorexia', 'relationship': 'associate', 'target entity': 'Sarcopenia', 'sentence': ["(1) Background: Appetite loss in older people, the 'Anorexia of Aging' (AA), is common, associated with under-nutrition, sarcopenia, and frailty and yet receives little attention."], 'source': ['Anorexia'], 'target': ['sarc

KeyboardInterrupt: 

What are the the key things in here? The node names are in the fields "source" and "target". We can also get the types of these nodes from the "source type" and "target type" fields. We also have the fielf "relationship" which is the type of the connection between the source and target. Everything else can be treated as attributes.

In [36]:
for relation in Relation_info.keys():
    Relationship = Relation_info[relation]['relationship']
    Source = Relation_info[relation]['source']
    Target = Relation_info[relation]['target']


RelationTypes = set()
for k in Relation_info.keys():
    RelationTypes.add(Relation_info[k]['relationship'])

print(RelationTypes)
with onto:
    RelationClasses = dict()
    for relation in RelationTypes:
        RelationClasses[relation] = type(relation, (ObjectProperty,), dict())
        print(f'Created relation class {RelationClasses[relation]}')
    

{'encoded', 'metastasize', 'include changes be', 'play role Besides', 'retained', 'certify', 'be similar between', 'reflected', 'cognitive dysfunction due', 'kill', 'stimulate matrix synthesis in', 'cloude', 'marked inhibition of', 'continue', 'reanalyzed', 'be serious risk for', 'uncoupled', 'keratinise', 'moving', 'elicited', 'try', 'rose', 'be up regulated in', 'be useful biomarker for', 'elect', 'require', 'extrapolated', 'centered', 'be frequent during', 'regimen of', 'intersect', 'index', 'provoked', 'co-stimulate', 'be In', 'disabled', 'switched', 'applied', 'view', 'enter', 'dichotomise', 'be actionable target in', 'intended', 'performed', 'inoculated', 'dominated', 'excised', 'objectify', 'compensate', 'have involve', 'related in', 'backed', 'binding between', 'saved', 'pose', 'vanish', 'increase levels of', 'proliferate', 'growing', 'play role besides', 'unravel', 'metabolise', 'differ accord', 'recognised', 'exceed', 'tried', 'set', 'be greater in', 'counteract', 'symptoms o

TypeError: __class__ assignment: 'ObjectPropertyClass' object layout differs from 'Mutation'

In [26]:
RelationDomainRange = dict.fromkeys(RelationTypes,None)
for r in RelationTypes:
    RelationDomainRange[r] = {'Domain':set(), 'Range':set()}

for k in edgekeys:
    relationtype = edges[k]['relationship']
    source = edges[k]['source type']
    target = edges[k]['target type']
    RelationDomainRange[relationtype]['Domain'].add(source[0])
    RelationDomainRange[relationtype]['Range'].add(target[0])

NameError: name 'RelationTypes' is not defined

print(f"There are {len(nodes)} and {len(edges)} edges")

Let's look at each of these in more detail. First, let's pick one of the nodes. Let's print the keys and choose one:

So what does this tell us? These are *annotations* of the node that tell us:
* The name of the **entity** (node)
* What the **type** of the node is (a gene, in this case)
* The **official full name** and **alias names** of the entity.
* A **URL** to the official record of the entity.
* A **description** of the entity.

Some entries tell us about the research papers that include information about the entity:

* The **number of articles** that mention this entity
* The **PMID** (pubmed identity) of the article that were used to get information about the entity. You can enter these numbers at [https://pubmed.ncbi.nlm.nih.gov](https://pubmed.ncbi.nlm.nih.gov) to get the papers themselves.
* A **sentence** containing the entity name from each of the articles.
* The **JT** (journal title) **TA** (journal title abbreviation) of each article.
* The **IF** (impact factor) and **IF5** (five year impact factor) of each of the journals.
* The **year** and **date** each of the articles was published.
* Information about mutations: **mutuation position** and **mutation alleles**.
* Some **external links** and the **MeSH ID** for the Medical Subject Headings database.
* Whether the entity is an **aging biomarker** or a **longenvity** biomarker.

Let's now look at one of the edges:

edgekeys = list(edges.keys())
edge = edges[edgekeys[50000]]
for k in edge.keys():
    print(f"{k}: {edge[k]}")

Perhaps the two key properties of the edge are the

* **source entity** and **target entity** which specify the *entity* property of the nodes that the edge connects. Let us check that these exist and see what they are:

source_node = nodes[edge['source entity']]
for k in source_node[0].keys():
    print(f"{k}: {source_node[0][k]}")
target_node = nodes[edge['target entity']]
for k in target_node[0].keys():
    print(f"{k}: {target_node[0][k]}")

The edge also has a

* **relationship**, which specifies how the two nodes are related.
* **source** and **target** attributes, which are alternative names for the entities and which we will not use.
* **source type** and **target type** which refer to the *type* atribute of the source and target nodes.
* A range of attributes related to the publications in which the relationship modelled by the edge is described (**PMID**, **DP**, **TI**,**TA**, **IF**, **IF5**).
* A list of **method**s, which we will not use.

## A deeper dive into the data

Let's do a deeper dive into the data now. We should easily be able to find out what the type of entity, the type of relation, and how many of each there are. This will require us to do a full pass through the data. The edges contain all of the information we need here so we can just iterate through those.



EntityTypes = set()
nodekeys = nodes.keys()
for k in nodekeys:
    EntityTypes.add(nodes[k][0]['type'])

print("The entity types are:")
for i in EntityTypes:
    print(i)

How many of each type are there?

EntityCount = dict.fromkeys(EntityTypes,0)
for k in nodekeys:
    EntityCount[nodes[k][0]['type']] += 1

print("The entity counts are:")
for i in EntityTypes:
    print(f"* {i}: {EntityCount[i]}")

Let's now repeat for the edges:

RelationTypes = set()
edgekeys = edges.keys()
for k in edgekeys:
    RelationTypes.add(edges[k]['relationship'])

#print("The entity types are:")
#for i in RelationTypes:
    #print(i)

How many of each type of relation?

RelationCount = dict.fromkeys(RelationTypes,0)
for k in edgekeys:
    RelationCount[edges[k]['relationship']] += 1

#print("The relation counts are:")
#for i in RelationTypes:
#    print(f"* {i}: {RelationCount[i]}")

Plot some graphs of these

import matplotlib.pyplot as plt
plt.pie(EntityCount.values(), labels=EntityCount.keys())
plt.title('Number of each type of Entity')

plt.pie(RelationCount.values(), labels=RelationCount.keys())
plt.title('Number of each type of Relation')

Also interesting to look at the number of times each type of node is connected to each other type of node.

EntityEntityCounts = dict.fromkeys(EntityTypes,0)
for k in EntityEntityCounts.keys():
    EntityEntityCounts[k] = dict.fromkeys(EntityTypes,0)

for k in edgekeys:
    source = edges[k]['source type'][0]
    target = edges[k]['target type'][0]
    EntityEntityCounts[source][target] += 1

for k in EntityTypes:
    print(EntityEntityCounts[k])

## Creating the Ontology

Now we are in a position to create the ontology. A major challenge here is that we need to do this dynamically. Our approach will be to store the classes in a `dictionary` from where they can still be called.

We will first create the entities.

from owlready2 import *
import types

onto = get_ontology("http://www.dummy.info/new.owl")

with onto:
    EntityClass = dict.fromkeys(EntityTypes, None)
    print(EntityClass)
    for t in EntityTypes:
       EntityClass[t] = types.new_class(t, (Thing,))
       print(EntityClass[t])
    print(EntityClass)

Now we create the relationships. For this, we need to determine the domain and range for each type of relation

In [ ]:
RelationDomainRange = dict.fromkeys(RelationTypes,None)
for r in RelationTypes:
    RelationDomainRange[r] = {'Domain':set(), 'Range':set()}

for k in edgekeys:
    relationtype = edges[k]['relationship']
    source = edges[k]['source type']
    target = edges[k]['target type']
    RelationDomainRange[relationtype]['Domain'].add(source[0])
    RelationDomainRange[relationtype]['Range'].add(target[0])

#for r in RelationDomainRange.keys():
#    print(f"{r}: {RelationDomainRange[r]}")


Now we have this, we can create the relations

Relations = dict.fromkeys(RelationTypes, None)
with onto:
    for t in RelationTypes:
        Relations[t] = types.new_class(t, (ObjectProperty,))
        domain = list(RelationDomainRange[t]['Domain'])
        range =  list(RelationDomainRange[t]['Range'])
        for d in domain:
            Relations[t].domain.append(EntityClass[d])
        for r in range:
            Relations[t].range.append(EntityClass[r])

Check this this looks sensible against the ones printed out above

print(Relations['conceived'].domain)
print(Relations['noticed'].range)

Let's now populate the graph. Start by adding the Entities

TheGraph = dict.fromkeys(nodekeys, None)
for n in nodekeys:
    etype = nodes[n][0]['type']
    TheGraph[n] = EntityClass[etype](name=n)
    TheGraph[n].rel = []

Now the edges

#TheGraph = dict.fromkeys(nodekeys, None)
#    nodetype = nodes[n][0]['type']
#for n in nodekeys:
#    TheGraph[n] = EntityClass[nodetype](name=n)
#    TheGraph[n].rel = []

#for k in edgekeys:
#
#    target = edges[k]['target entity']
#    source = edges[k]['source entity']
#    TheGraph[source].rel.append(TheGraph[target])

Get the edges for one node.

#TheGraph['MLH1'].rel

Which nodes has the most connections from it?
Which node has the most connections to it?

#max_links = 0
#for n in TheGraph.keys():
#    if len(TheGraph[n].rel)> max_links:
#        max_links = len(TheGraph[n].rel)
#        max_node = n

#print(f"{max_node} has the most ({max_links}) outgoing links")

#IncomingLinks = dict.fromkeys(nodekeys,0)
#for n in TheGraph.keys():
#    targets = TheGraph[n].rel
#    for t in targets:
#        IncomingLinks[t.name] += 1

#print(f"{max(IncomingLinks, key=lambda k: IncomingLinks[k])} has the most incoming links ({max(IncomingLinks.values())})")